In [2]:
import pandas as pd
import numpy as np
import joblib
import glob
import hashlib
from sklearn.decomposition import TruncatedSVD
import os
import re

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle

# Import all modular extractors and dictionaries
from GUI_app.bioprocessing_eda import (
    load_and_clean_data,
    compute_esm_embeddings,
    compute_georgiev_features,
    GEORGIEV_DICT, 
    compute_aac_features,
    compute_aaindex_features,
    SelfContainedTargetTransformRegressor,
    load_aaindex
)

def get_required_feature_types(features):
    required = []
    for f in features:
        f_str = str(f).upper()
        if "AAC" in f_str and 'AAC' not in required: required.append('AAC')
        if "AAINDEX" in f_str and 'AAindex' not in required: required.append('AAindex')
        if "GEORGIEV" in f_str and 'Georgiev' not in required: required.append('Georgiev')
        if "ESM" in f_str and 'ESM' not in required: required.append('ESM')
        if "SVD" in f_str and 'ESM_SVD' not in required: required.append('ESM_SVD')
    return required

def get_esm_name(features):
    if any("ESM_Massive_3B_" in f for f in features): return "facebook/esm2_t36_3B_UR50D"
    if any("ESM_Big_650M_" in f for f in features): return "facebook/esm2_t33_650M_UR50D"
    if any("ESM_Large_150M_" in f for f in features): return "facebook/esm2_t30_150M_UR50D"
    if any("ESM_Medium_35M_" in f for f in features): return "facebook/esm2_t12_35M_UR50D"
    return "facebook/esm2_t6_8M_UR50D"

def get_esm_tag(features):
    if any("ESM_Massive_3B_" in f for f in features): return "ESM_Massive_3B"
    if any("ESM_Big_650M_" in f for f in features): return "ESM_Big_650M"
    if any("ESM_Large_150M_" in f for f in features): return "ESM_Large_150M"
    if any("ESM_Medium_35M_" in f for f in features): return "ESM_Medium_35M"
    if any("ESM_Small_8M_" in f for f in features): return "ESM_Small_8M"
    return "ESM"

def get_required_regions(features):
    regions = set()
    for f in features:
        f_str = str(f).upper()
        if 'VH' in f_str: regions.add('CD3_VH')
        if 'VL' in f_str: regions.add('CD3_VL')
        if 'SCFV' in f_str: regions.add('scFv')
    return list(regions) if regions else ['CD3_VH', 'CD3_VL', 'scFv']

def prepare_inference_data(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    df = df.rename(columns={
        'CD3 VH_HCK': 'CD3_VH', 
        'CD3 VL_HCK': 'CD3_VL', 
        'CD3_VH_HCK': 'CD3_VH', 
        'CD3_VL_HCK': 'CD3_VL'
    })
    
    if 'CD3_VH' in df.columns and 'CD3_VL' in df.columns:
        def build_fv(r):
            vh = str(r['CD3_VH']).strip().upper() if pd.notna(r['CD3_VH']) else 'NAN'
            vl = str(r['CD3_VL']).strip().upper() if pd.notna(r['CD3_VL']) else 'NAN'
            if vh == 'NAN' or vl == 'NAN': return 'NAN'
            
            linker = ''
            if 'G4S Linker2_HCK' in df.columns and pd.notna(r['G4S Linker2_HCK']):
                val = str(r['G4S Linker2_HCK']).strip().upper()
                if val not in ['NAN', 'NONE']: linker = val
            return vh + linker + vl
            
        df['scFv'] = df.apply(build_fv, axis=1)
    return df

def fit_base_svd(df_base, region_col, esm_model_name, esm_tag, prefix, target_indices=None):
    print(f" ⚙️ Fitting SVD for {prefix} using {esm_model_name}...")
    valid_seqs = df_base[region_col].fillna("").astype(str)
    _, raw_matrix = compute_esm_embeddings(
        sequences=valid_seqs, esm_model_name=esm_model_name, 
        prefix=prefix, esm_tag=esm_tag, target_indices=target_indices
    )
    svd = TruncatedSVD(n_components=50, random_state=42)
    svd.fit(raw_matrix)
    return svd

def extract_features_for_model(df_test, pkg, esm_name, esm_tag, ftypes, target_regions, fitted_svds, my_target_indices=None):
    df_features = pd.DataFrame(index=df_test.index)
    
    aaindex_db = None
    if 'AAindex' in ftypes:
        aaindex_db, _ = load_aaindex()
    
    for col in target_regions:
        valid_seqs = df_test[col].fillna("").astype(str)
        
        extraction_passes = [(False, f"{col}_", None)]
        if my_target_indices and col in my_target_indices:
            semantic_tag, target_indices = my_target_indices[col]
            short_hash = hashlib.md5(str(target_indices).encode('utf-8')).hexdigest()[:6]
            
            t_prefix = f"{col}_{semantic_tag}_{short_hash}_i-" 
            extraction_passes.append((True, t_prefix, target_indices))
            
        for is_targeted, prefix, t_idx in extraction_passes:
            
            # 🌟 SNIPER LOGIC: Only extract if the loaded model explicitly requires this exact prefix!
            if not any(f.startswith(prefix) for f in pkg['features']):
                continue
                
            scope_tag = f"🎯 TARGETED ({semantic_tag})" if is_targeted else "🌍 GLOBAL"
                
            if 'ESM' in ftypes:
                print(f" ⚙️ [{scope_tag}] Computing ESM for {col}...")
                esm_dict, raw_matrix = compute_esm_embeddings(
                    sequences=valid_seqs, esm_model_name=esm_name, 
                    prefix=prefix, esm_tag=esm_tag, target_indices=t_idx
                )
                if 'ESM_SVD' in ftypes:
                    svd_transformed = fitted_svds[prefix].transform(raw_matrix)
                    for i in range(svd_transformed.shape[1]):
                        df_features[f"{prefix}{esm_tag}_SVD50_{i}"] = svd_transformed[:, i]
                else:
                    df_features = pd.concat([df_features, pd.DataFrame(esm_dict, index=df_test.index)], axis=1)
                    
            if 'Georgiev' in ftypes:
                print(f" ⚙️ [{scope_tag}] Computing Georgiev for {col}...")
                geo_out = compute_georgiev_features(valid_seqs, GEORGIEV_DICT, prefix=prefix, target_indices=t_idx)
                df_features = pd.concat([df_features, pd.DataFrame(geo_out, index=df_test.index)], axis=1)

            if 'AAC' in ftypes:
                print(f" ⚙️ [{scope_tag}] Computing AAC for {col}...")
                aac_out = compute_aac_features(valid_seqs, prefix=prefix, target_indices=t_idx)
                df_features = pd.concat([df_features, pd.DataFrame(aac_out, index=df_test.index)], axis=1)
                
            if 'AAindex' in ftypes:
                print(f" ⚙️ [{scope_tag}] Computing AAindex for {col}...")
                idx_out = compute_aaindex_features(valid_seqs, aaindex_db, prefix=prefix, target_indices=t_idx)
                df_features = pd.concat([df_features, pd.DataFrame(idx_out, index=df_test.index)], axis=1)

    df_features = df_features.loc[:, ~df_features.columns.duplicated()]

    # ==========================================
    # 🌟 NEW: TABULAR FEATURE SWEEPER
    # ==========================================
    # Loop through exactly what the trained model expects.
    # If we didn't calculate it mathematically, pull it straight from the CSV.
    for required_feature in pkg['features']:
        if required_feature not in df_features.columns:
            if required_feature in df_test.columns:
                df_features[required_feature] = df_test[required_feature].values
                print(f" 📥 [TABULAR] Loaded pre-computed CSV column: '{required_feature}'")
            else:
                print(f" ⚠️ CRITICAL WARNING: Model requires '{required_feature}', but it is missing from both the calculations and your unseen.csv!")

    return df_features

def evaluate_and_save_excel(df_test, predictions, actual_col_keyword, output_excel, target_name, percent_range):
    """
    Saves the multi-sheet Excel for a single model and returns the metrics for the combined plot.
    """
    print(f"\n--- Evaluating {target_name} Performance ---")
    
    seq_ids = df_test['ID'] if 'ID' in df_test.columns else (df_test['Samples'] if 'Samples' in df_test.columns else df_test.index)
    base_results_df = pd.DataFrame({
        'Sequence_ID': seq_ids,
        f'Predicted_{target_name}': predictions
    })
    
    actual_col = next((c for c in df_test.columns if actual_col_keyword.lower() in c.lower()), None)
    
    if not actual_col:
        print(f"⚠️ Actual column for '{actual_col_keyword}' not found. Skipping evaluation.")
        return [], [], 0.0
        
    base_results_df[f'Actual_{target_name}'] = df_test[actual_col]
    
    base_results_df['Actual_Rank'] = base_results_df[f'Actual_{target_name}'].rank(method='min')
    base_results_df['Predicted_Rank'] = base_results_df[f'Predicted_{target_name}'].rank(method='min')
    
    overall_spearman = base_results_df[f'Actual_{target_name}'].corr(base_results_df[f'Predicted_{target_name}'], method='spearman')
    print(f"Overall Spearman Rank Correlation on Unseen Data: {overall_spearman:.3f}")
    
    summary_data = []
    hit_rates_for_plot = []
    thresholds_for_plot = [int(p * 100) for p in percent_range]

    # Write multi-sheet Excel
    with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
        for p in percent_range:
            top_k = max(1, int(len(base_results_df) * p))
            
            sheet_df = base_results_df.copy()
            sheet_df['Is_Hit'] = (sheet_df['Actual_Rank'] <= top_k) & (sheet_df['Predicted_Rank'] <= top_k)
            
            hits = sheet_df['Is_Hit'].sum()
            hit_percentage = (hits / top_k) * 100 if top_k > 0 else 0
            hit_rates_for_plot.append(hit_percentage)
            
            summary_data.append({
                'Top Tier Target': f"Top {int(p*100)}% (N={top_k})",
                'Hit Rate (Fraction & %%)': f"{hits}/{top_k} ({hit_percentage:.1f}%)"
            })
            
            sheet_name = f"Top_{int(p*100)}_Percent"
            sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)
            
        summary_df = pd.DataFrame(summary_data)
        summary_df.loc[len(summary_df)] = ["", ""]
        summary_df.loc[len(summary_df)] = ["Overall Spearman Correlation", f"{overall_spearman:.3f}"]
        
        summary_df.to_excel(writer, sheet_name='Summary', index=False)
        workbook = writer.book
        summary_sheet = workbook['Summary']
        workbook._sheets.remove(summary_sheet)
        workbook._sheets.insert(0, summary_sheet)

    print(f"🚀 Excel results saved to {output_excel}")
    return thresholds_for_plot, hit_rates_for_plot, overall_spearman

def plot_combined_model_comparison(target_name, results_dict, thresholds, output_filename):
    """
    Generates a master visual dashboard comparing unseen inference performance.
    Uses a Heatmap aligned perfectly with an inverted Spearman bar chart.
    """
    if not results_dict:
        return
        
    sns.set_theme(style="whitegrid")
    
    # Use gridspec to make the heatmap slightly wider to accommodate the y-axis labels
    fig, axes = plt.subplots(1, 2, figsize=(20, 8), gridspec_kw={'width_ratios': [1.3, 1]})
    fig.suptitle(f'Unseen Data Inference Comparison: {target_name}', fontsize=18, fontweight='bold', y=1.02)
    
    # 🌟 FIX: Sort descending so the BEST model is at index 0 (the top)
    sorted_items = sorted(results_dict.items(), key=lambda x: x[1]['spearman'], reverse=True)
    model_names = [item[0] for item in sorted_items]
    spearmans = [item[1]['spearman'] for item in sorted_items]
    
    # Extract hit rates into a 2D matrix
    hit_rate_matrix = np.array([item[1]['hit_rates'] for item in sorted_items])
    
    # --- PANEL 1: Enrichment Hit Rate Heatmap ---
    sns.heatmap(hit_rate_matrix, annot=True, fmt=".1f", cmap="Blues", 
                cbar_kws={'label': 'True Hits Found (%)'}, ax=axes[0],
                linewidths=1, linecolor='white')
    
    # Format Heatmap Axes
    axes[0].set_title('Enrichment Accuracy Across Thresholds', fontsize=14, pad=15)
    axes[0].set_xlabel('Top Tier Threshold Evaluated (%)', fontsize=12, labelpad=10)
    axes[0].set_xticks(np.arange(len(thresholds)) + 0.5)
    axes[0].set_xticklabels([f"{t}%" for t in thresholds], fontsize=11)
    
    axes[0].set_yticks(np.arange(len(model_names)) + 0.5)
    axes[0].set_yticklabels(model_names, rotation=0, fontsize=10)
    
    # --- PANEL 2: Spearman Leaderboard (Bar Chart) ---
    bar_colors = sns.color_palette("Greens_r", len(model_names))
    
    # 🌟 FIX 1: Shift the bar centers to 0.5, 1.5, 2.5 to perfectly match the heatmap's internal grid
    y_positions = np.arange(len(model_names)) + 0.5
    
    # 🌟 FIX 2: Set height=0.6 (default is 0.8) to shrink the bars and increase the visual gap
    axes[1].barh(y_positions, spearmans, height=0.6, color=bar_colors, edgecolor='black', alpha=0.9)
    
    # 🌟 FIX 3: Force the Bar Chart to literally copy the Heatmap's exact Y-axis limits.
    # (This also automatically inverts the axis for us, so we no longer need invert_yaxis()!)
    axes[1].set_ylim(axes[0].get_ylim())
    
    # Format Bar Chart Axes
    axes[1].set_title('Out-of-Sample Spearman Rank Correlation', fontsize=14, pad=15)
    axes[1].set_xlabel('Spearman Correlation Score', fontsize=12, labelpad=10)
    axes[1].set_xlim(0, max(max(spearmans) + 0.1, 0.75))
    
    # Hide the Y-axis labels on the bar chart because they perfectly align with the Heatmap
    axes[1].set_yticks(y_positions)
    axes[1].set_yticklabels([]) 
    
    # Print the exact score on the bars (aligned to the new y_positions)
    for i, v in enumerate(spearmans):
        axes[1].text(v + 0.01, y_positions[i], f"{v:.3f}", va='center', fontweight='bold', fontsize=11)
        
    plt.tight_layout()
    plt.savefig(output_filename, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"\n📊 Master Dashboard successfully saved to: {output_filename}")

def get_clean_model_name(basename):
    """
    Parses a long joblib filename and converts it into a clean, compact abbreviation.
    Uses regex and masking to safely isolate structural regions from features.
    """
    # 1. Isolate the core string by removing the XGBoost/SVR prefix
    tag = basename
    if 'CD3_' in tag:
        tag = 'CD3_' + tag.split('CD3_', 1)[1]
    elif 'scFv' in tag:
        tag = 'scFv' + tag.split('scFv', 1)[1]
        
    # 2. Safely separate structural regions from features using regex
    match = re.match(r'^((?:CD3_VH|CD3_VL|scFv)(?:-(?:CD3_VH|CD3_VL|scFv))*?)_(.*)$', tag)
    
    if match:
        subregions = match.group(1)
        features = match.group(2)
    else:
        subregions = tag
        features = ""
        
    # 3. Clean up the structural subregions
    subregions = subregions.replace("CD3_VH", "VH").replace("CD3_VL", "VL").replace("-", "+")
    
    if features:
        # 4. Protect features with internal hyphens FIRST
        features = features.replace("50-50_HCCF_Titer", "Titer")
        features = features.replace("i-", "INTERFACE_PROTECT_")
        
        # 5. Now it is safe to swap the feature delimiter from hyphen to plus
        features = features.replace("-", "+")
        
        # 6. Restore the protected 'i-' prefix
        features = features.replace("INTERFACE_PROTECT_", "i-")
        
        # 7. Apply all remaining abbreviations safely
        mapping = {
            "Targeted_": "i-", # Kept just in case you ever load an older model
            "Georgiev": "Geo",
            "AAindex": "AAidx",
            "ESM_Big_650M_SVD50": "ESM(SVD)",
            "ESM_Big_650M": "ESM650",
            "ESM_Medium_35M": "ESM35",
            "ESM_Small_8M": "ESM8",
            "AntiBERTy": "ABerty",
            "AbLang2_Paired": "AbL2",
            "Propermab": "3D",
            "Delta_G_Rank1": "dG",
            "VH_VL_Log10_Kd": "Kd"
        }
        for old, new in mapping.items():
            features = features.replace(old, new)
            
        short_name = f"{subregions} | {features}"
    else:
        short_name = subregions
        
    # Final safety truncate just in case it's still too long for the chart
    if len(short_name) > 40:
        short_name = short_name[:37] + "..."
        
    return short_name

def main():
    base_csv_poly = "data/tubespin_subset.csv" 
    base_csv_hmw = "data/50-50_sequences_subset.csv" 
    
    percent_range = [0.20, 0.30, 0.40, 0.50, 0.60]
    
    my_target_indices = {
        'CD3_VH': ('Interface_looseness', [34, 36, 38, 42, 43, 44, 45, 46, 49, 60, 61, 62, 63, 96, 101, 102, 103, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117]),
        'CD3_VL': ('Interface_looseness', [30, 33, 34, 35, 36, 37, 39, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 54, 55, 56, 57, 88, 90, 92, 94, 95, 96, 97, 98, 99, 100, 101])
    }
    
    # ==========================================
    # 1. POLYREACTIVITY PIPELINE
    # ==========================================
    poly_model_paths = glob.glob("GUI_app/trained_models/*ELISA_Polyreactivity_Excell*.joblib")
    
    if poly_model_paths:
        print(f"\n🔹 Found {len(poly_model_paths)} Polyreactivity models! Running batch inference...")
        poly_holdout_csv = "data/tubespin_unseen.csv" 
        df_base_poly = prepare_inference_data(base_csv_poly)
        df_test_poly = prepare_inference_data(poly_holdout_csv)
        poly_results = {}
        poly_thresholds = []
        
        for path in poly_model_paths:
            basename = os.path.splitext(os.path.basename(path))[0]
            output_excel = f"model_comparison/{basename}_Predictions.xlsx"
            
            # Extract just the structural & feature combination for a clean plot legend
            clean_name = get_clean_model_name(basename)
            
            pkg = joblib.load(path)
            ftypes = get_required_feature_types(pkg['features'])
            esm_name = get_esm_name(pkg['features'])
            esm_tag = get_esm_tag(pkg['features'])
            regions = get_required_regions(pkg['features'])
            
            fitted_svds = {}
            if 'ESM_SVD' in ftypes:
                for reg in regions:
                    fitted_svds[f"{reg}_"] = fit_base_svd(df_base_poly, reg, esm_name, esm_tag, prefix=f"{reg}_")
                    if reg in my_target_indices:
                        sem_tag, t_idx = my_target_indices[reg]
                        short_hash = hashlib.md5(str(t_idx).encode('utf-8')).hexdigest()[:6]
                        
                        # 🌟 CHANGED: Replaced '_Targeted_' with the new 'i-' suffix format
                        t_prefix = f"{reg}_{sem_tag}_{short_hash}_i-"
                        
                        fitted_svds[t_prefix] = fit_base_svd(df_base_poly, reg, esm_name, esm_tag, prefix=t_prefix, target_indices=t_idx)

            X = extract_features_for_model(df_test_poly, pkg, esm_name, esm_tag, ftypes, regions, fitted_svds, my_target_indices)
            pred = pkg['model'].predict(X[pkg['features']]).flatten()
            
            poly_thresholds, hit_rates, spearman = evaluate_and_save_excel(df_test_poly, pred, 'Poly', output_excel, 'Poly', percent_range)
            poly_results[clean_name] = {'hit_rates': hit_rates, 'spearman': spearman}
            
        plot_combined_model_comparison('Polyreactivity', poly_results, poly_thresholds, "model_comparison/Combined_Polyreactivity_Comparison.png")


    # ==========================================
    # 2. HMW PIPELINE
    # ==========================================
    hmw_model_paths = glob.glob("GUI_app/trained_models/*HMW*.joblib")
    
    if hmw_model_paths:
        print(f"\n🔹 Found {len(hmw_model_paths)} HMW models! Running batch inference...")
        hmw_holdout_csv = "data/50-50_sequences_unseen.csv" 
        df_base_hmw = prepare_inference_data(base_csv_hmw)
        df_test_hmw = prepare_inference_data(hmw_holdout_csv)
        hmw_results = {}
        hmw_thresholds = []
        
        for path in hmw_model_paths:
            basename = os.path.splitext(os.path.basename(path))[0]
            output_excel = f"model_comparison/{basename}_Predictions.xlsx"
            
            # Extract just the structural & feature combination for a clean plot legend
            clean_name = get_clean_model_name(basename)
        
            pkg = joblib.load(path)
            ftypes = get_required_feature_types(pkg['features'])
            esm_name = get_esm_name(pkg['features'])
            esm_tag = get_esm_tag(pkg['features'])
            regions = get_required_regions(pkg['features'])
            
            fitted_svds = {}
            if 'ESM_SVD' in ftypes:
                for reg in regions:
                    fitted_svds[f"{reg}_"] = fit_base_svd(df_base_hmw, reg, esm_name, esm_tag, prefix=f"{reg}_")
                    if reg in my_target_indices:
                        sem_tag, t_idx = my_target_indices[reg]
                        short_hash = hashlib.md5(str(t_idx).encode('utf-8')).hexdigest()[:6]
                        
                        # 🌟 CHANGED: Replaced '_Targeted_' with the new 'i-' suffix format
                        t_prefix = f"{reg}_{sem_tag}_{short_hash}_i-"
                        
                        fitted_svds[t_prefix] = fit_base_svd(df_base_poly, reg, esm_name, esm_tag, prefix=t_prefix, target_indices=t_idx)

            X = extract_features_for_model(df_test_hmw, pkg, esm_name, esm_tag, ftypes, regions, fitted_svds, my_target_indices)
            pred = pkg['model'].predict(X[pkg['features']]).flatten()
            
            hmw_thresholds, hit_rates, spearman = evaluate_and_save_excel(df_test_hmw, pred, 'HMW', output_excel, 'HMW', percent_range)
            hmw_results[clean_name] = {'hit_rates': hit_rates, 'spearman': spearman}
            
        plot_combined_model_comparison('HMW', hmw_results, hmw_thresholds, "model_comparison/Combined_HMW_Comparison.png")


if __name__ == "__main__":
    main()


🔹 Found 2 Polyreactivity models! Running batch inference...
 ⚙️ [🌍 GLOBAL] Computing Georgiev for CD3_VL...
 ⚙️ [🌍 GLOBAL] Computing Georgiev for CD3_VH...

--- Evaluating Poly Performance ---
Overall Spearman Rank Correlation on Unseen Data: 0.733
🚀 Excel results saved to model_comparison/Production_global_SVR_SUBSET_ELISA_Polyreactivity_Excell_CD3_VH-CD3_VL_Georgiev_Predictions.xlsx
 ⚙️ [🌍 GLOBAL] Computing Georgiev for CD3_VL...
 ⚙️ [🌍 GLOBAL] Computing AAC for CD3_VL...
 ⚙️ [🌍 GLOBAL] Computing Georgiev for CD3_VH...
 ⚙️ [🌍 GLOBAL] Computing AAC for CD3_VH...

--- Evaluating Poly Performance ---
Overall Spearman Rank Correlation on Unseen Data: 0.685
🚀 Excel results saved to model_comparison/Production_global_SVR_SUBSET_ELISA_Polyreactivity_Excell_CD3_VH-CD3_VL_AAC-Georgiev_Predictions.xlsx

📊 Master Dashboard successfully saved to: model_comparison/Combined_Polyreactivity_Comparison.png

🔹 Found 8 HMW models! Running batch inference...
 ⚙️ [🌍 GLOBAL] Computing ESM for CD3_VL...
 